# Notebook 07: Kaggle-Only Baseline Evaluation on Gold Benchmark

## Purpose

This notebook evaluates the **Kaggle-only baseline RAG pipeline** on a verified gold benchmark of 50 Turkish legal questions.

**Scope:**
- Load a 50-question gold benchmark CSV with verified answers and sources
- Reuse the Kaggle-only retrieval pipeline (dense + BM25 + hybrid + reranker)
- Reuse the Kaggle-only LLM generation pipeline
- For each benchmark question, run the full Kaggle-only RAG pipeline and collect predictions
- Evaluate retrieval quality (source matching, article matching)
- Evaluate answer quality (ROUGE-L, BLEU, token F1)
- Perform error analysis on worst cases, grouped by legal domain and difficulty
- Save detailed results with Kaggle-prefixed output filenames

**Output (Kaggle-only naming):**
- `kaggle_gold50_rag_results.csv` - Full detailed results
- `kaggle_gold50_metrics_summary.json` - Aggregate metrics
- `kaggle_gold50_error_analysis.csv` - Error cases
- `kaggle_gold50_domain_breakdown.csv` - Performance by legal domain
- `kaggle_gold50_difficulty_breakdown.csv` - Performance by difficulty level

**Note:** This is a benchmark evaluation notebook using the Kaggle-only baseline, not training. The gold benchmark is locked as test input.

---

# Section 1: Configuration and Paths

In [ ]:
import os
import sys

# ===================== GOOGLE COLAB DETECTION & MOUNT =====================
IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted at /content/drive")

# ===================== PATHS - KAGGLE-ONLY BASELINE =====================
# Google Drive path for Colab execution
PROJECT_ROOT = "/content/drive/My Drive/nlp-rag-project"

# Gold benchmark
GOLD_BENCHMARK_PATH = os.path.join(PROJECT_ROOT, "data", "gold", "turkish_legal_gold50_draft.csv")

# Retrieval corpus and artifacts (KAGGLE-ONLY)
RETRIEVAL_CORPUS_PATH = os.path.join(PROJECT_ROOT, "data", "retrieval", "kaggle_retrieval_corpus.csv")
DENSE_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "dense_retrieval")
RERANKER_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "reranker")

# Evaluation output
EVAL_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "evaluation")
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

# ===================== MODELS =====================
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GENERATION_MODEL_NAME = "google/gemma-2-9b-it"

# ===================== RETRIEVAL CONFIG =====================
TOP_K_FINAL = 3
DENSE_CANDIDATES = 20
BM25_CANDIDATES = 20
HYBRID_CANDIDATES = 20
ALPHA = 0.5

# ===================== GENERATION CONFIG =====================
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.2
TOP_P = 0.9
TOP_K = 40

print(f"Project root: {PROJECT_ROOT}")
print(f"Gold benchmark: {GOLD_BENCHMARK_PATH}")
print(f"Evaluation output: {EVAL_OUTPUT_DIR}")
print(f"Retrieval corpus: {RETRIEVAL_CORPUS_PATH}")
print(f"\n✓ All Kaggle-only baseline paths configured.")

# Section 2: Install and Import Dependencies

In [ ]:
# Install dependencies if needed
!pip install -q pandas numpy tqdm scikit-learn rouge-score
!pip install -q faiss-cpu rank-bm25 sentence-transformers transformers torch accelerate

print("Dependencies installed.")

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from typing import List, Dict, Tuple, Optional
from tqdm import tqdm

# NLP/ML libraries
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Metrics
from rouge_score import rouge_scorer
from nltk.translate import bleu_score
from nltk.tokenize import word_tokenize
from sklearn.metrics import precision_recall_fscore_support

print("All dependencies imported successfully.")

# Section 3: Load Gold Benchmark

In [ ]:
# Load gold benchmark
print(f"Loading gold benchmark from: {GOLD_BENCHMARK_PATH}")

gold_df = pd.read_csv(GOLD_BENCHMARK_PATH, encoding="cp1254")

print(f"\nRaw columns loaded: {gold_df.columns.tolist()}")

# ===================== SCHEMA NORMALIZATION =====================
# Rename actual column names to expected schema
column_mapping = {
    'suggested_source_law': 'gold_source',
    'suggested_article_reference': 'gold_article'
}

for old_col, new_col in column_mapping.items():
    if old_col in gold_df.columns:
        gold_df.rename(columns={old_col: new_col}, inplace=True)
        print(f"✓ Renamed '{old_col}' → '{new_col}'")

# Ensure required columns exist
required_cols = ['question', 'gold_answer', 'gold_source', 'gold_article', 'legal_domain', 'difficulty']
for col in required_cols:
    if col not in gold_df.columns:
        print(f"WARNING: Column '{col}' not found, creating empty column")
        gold_df[col] = None

# Safe fillna for string columns
for col in ['question', 'gold_answer', 'gold_source', 'gold_article', 'legal_domain', 'difficulty']:
    if col in gold_df.columns:
        gold_df[col] = gold_df[col].fillna('').astype(str)

print(f"\n✓ Gold benchmark loaded and normalized:")
print(f"  Shape: {gold_df.shape}")
print(f"  Columns: {gold_df.columns.tolist()}")

In [ ]:
# ===================== VALIDATION & DEBUG =====================
print("\n" + "="*80)
print("CSV SCHEMA VALIDATION")
print("="*80)

print(f"\nFirst 3 rows - question, gold_answer, gold_source, gold_article:")
for idx, row in gold_df.head(3).iterrows():
    print(f"\n[Row {idx}]")
    print(f"  question: {row['question'][:60]}...")
    print(f"  gold_answer: {row['gold_answer'][:60]}...")
    print(f"  gold_source: {row['gold_source']}")
    print(f"  gold_article: {row['gold_article']}")

print(f"\nNon-null counts:")
for col in ['question', 'gold_answer', 'gold_source', 'gold_article', 'legal_domain', 'difficulty']:
    non_null = (gold_df[col] != '').sum()
    print(f"  {col}: {non_null}/{len(gold_df)}")

print("\n" + "="*80)

# Section 4: Load Retrieval Corpus and Artifacts

In [ ]:
# Load retrieval corpus
print(f"Loading retrieval corpus...")
retrieval_corpus = pd.read_csv(RETRIEVAL_CORPUS_PATH)
print(f"✓ Corpus shape: {retrieval_corpus.shape}")

# Load dense embeddings (KAGGLE-ONLY)
print(f"\nLoading Kaggle-only dense retrieval artifacts...")
embeddings_path = os.path.join(DENSE_OUTPUT_DIR, "kaggle_dense_embeddings.npy")
corpus_embeddings = np.load(embeddings_path)
print(f"✓ Embeddings shape: {corpus_embeddings.shape}")

# Load FAISS index (KAGGLE-ONLY)
faiss_index_path = os.path.join(DENSE_OUTPUT_DIR, "kaggle_faiss_index.index")
faiss_index = faiss.read_index(faiss_index_path)
print(f"✓ FAISS index loaded, ntotal={faiss_index.ntotal}")

# Load row mapping (KAGGLE-ONLY)
row_mapping_path = os.path.join(DENSE_OUTPUT_DIR, "kaggle_retrieval_row_mapping.csv")
row_mapping = pd.read_csv(row_mapping_path)
print(f"✓ Row mapping shape: {row_mapping.shape}")

# Section 5: Load Retrieval Models

In [ ]:
# Load embedding model
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✓ Embedding model loaded")

# Load reranker
print(f"\nLoading reranker: {RERANKER_MODEL_NAME}")
reranker = CrossEncoder(RERANKER_MODEL_NAME)
print(f"✓ Reranker loaded")

# Section 6: Load Generation Model

In [ ]:
# Load tokenizer and generation model
print(f"Loading generation model: {GENERATION_MODEL_NAME}")
print(f"(This may take 1-2 minutes)\n")

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
print(f"✓ Tokenizer loaded")

model = AutoModelForCausalLM.from_pretrained(
    GENERATION_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Model loaded")
print(f"  Device map: auto (GPU if available)")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# Section 7: Retrieval Pipeline Functions

In [ ]:
def retrieve_dense_candidates(
    query: str,
    embedding_model: SentenceTransformer,
    faiss_index,
    retrieval_corpus: pd.DataFrame,
    row_mapping: pd.DataFrame,
    k: int
) -> pd.DataFrame:
    """Retrieve dense candidates using FAISS."""
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    query_embedding = query_embedding.reshape(1, -1).astype(np.float32)
    faiss.normalize_L2(query_embedding)
    
    distances, indices = faiss_index.search(query_embedding, k)
    scores = distances[0]
    indices = indices[0]
    
    corpus_indices = row_mapping.loc[indices, 'corpus_row_index'].values
    results = retrieval_corpus.iloc[corpus_indices].copy()
    results['dense_score'] = scores
    
    return results.reset_index(drop=True)


def retrieve_bm25_candidates(
    query: str,
    retrieval_corpus: pd.DataFrame,
    k: int
) -> pd.DataFrame:
    """Retrieve BM25 candidates."""
    corpus_texts = retrieval_corpus['chunk_text'].fillna('').str.lower().str.split().tolist()
    bm25 = BM25Okapi(corpus_texts)
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    
    top_indices = np.argsort(scores)[::-1][:k]
    results = retrieval_corpus.iloc[top_indices].copy()
    results['bm25_score'] = scores[top_indices]
    
    return results.reset_index(drop=True)


def retrieve_hybrid_candidates(
    query: str,
    embedding_model: SentenceTransformer,
    faiss_index,
    retrieval_corpus: pd.DataFrame,
    row_mapping: pd.DataFrame,
    dense_k: int,
    bm25_k: int,
    alpha: float
) -> pd.DataFrame:
    """Retrieve hybrid candidates: dense + BM25 fusion."""
    dense_results = retrieve_dense_candidates(
        query, embedding_model, faiss_index, retrieval_corpus, row_mapping, dense_k
    )
    bm25_results = retrieve_bm25_candidates(query, retrieval_corpus, bm25_k)
    
    merged = pd.concat([dense_results, bm25_results], ignore_index=True)
    merged = merged.drop_duplicates(subset=['chunk_id'], keep='first').reset_index(drop=True)
    
    merged['dense_score'] = merged['dense_score'].fillna(0.0)
    merged['bm25_score'] = merged['bm25_score'].fillna(0.0)
    
    if merged['dense_score'].max() > 0:
        dense_norm = (merged['dense_score'] - merged['dense_score'].min()) / (merged['dense_score'].max() - merged['dense_score'].min())
    else:
        dense_norm = pd.Series(0.0, index=merged.index)
    
    if merged['bm25_score'].max() > 0:
        bm25_norm = (merged['bm25_score'] - merged['bm25_score'].min()) / (merged['bm25_score'].max() - merged['bm25_score'].min())
    else:
        bm25_norm = pd.Series(0.0, index=merged.index)
    
    merged['hybrid_score'] = alpha * dense_norm + (1 - alpha) * bm25_norm
    merged = merged.sort_values('hybrid_score', ascending=False).reset_index(drop=True)
    
    return merged


def rerank_candidates(
    query: str,
    candidate_df: pd.DataFrame,
    reranker: CrossEncoder,
    top_k: int
) -> pd.DataFrame:
    """Rerank candidates using cross-encoder."""
    if len(candidate_df) == 0:
        return pd.DataFrame()
    
    chunks = candidate_df['chunk_text'].tolist()
    pairs = [[query, chunk] for chunk in chunks]
    scores = reranker.predict(pairs)
    
    result_df = candidate_df.copy()
    result_df['reranker_score'] = scores
    result_df = result_df.sort_values('reranker_score', ascending=False).reset_index(drop=True)
    result_df['rank'] = range(1, len(result_df) + 1)
    
    return result_df.head(top_k)


def get_final_context(query: str, top_k: int) -> pd.DataFrame:
    """Full retrieval pipeline: hybrid + reranker."""
    hybrid = retrieve_hybrid_candidates(
        query,
        embedding_model,
        faiss_index,
        retrieval_corpus,
        row_mapping,
        dense_k=DENSE_CANDIDATES,
        bm25_k=BM25_CANDIDATES,
        alpha=ALPHA
    )
    
    reranked = rerank_candidates(query, hybrid.head(HYBRID_CANDIDATES), reranker, top_k)
    
    return reranked


print("Retrieval functions defined.")

# Section 8: Answer Generation Function

In [ ]:
def build_rag_prompt(query: str, context_chunks: pd.DataFrame) -> Tuple[str, List[str]]:
    """Build retrieval-aware prompt."""
    system_instruction = """Sen Türk hukuk asistanısın. Sana verilen bağlam bilgisindeseniz kalmak zorundasın.
Dış bilgi KULLANAMAZSUN. Bağlam yetersiz veya belirsizse açıkça belirt.
Cevabını kısa, bilgilendirici ve hukuki tarz da verin. Sonunda destek kaynakları listele."""
    
    context_text = ""
    source_names = []
    used_count = 0
    
    for idx, row in context_chunks.iterrows():
        source = str(row.get('source', 'Bilinmeyen Kaynak')).strip()
        chunk = str(row.get('chunk_text', '')).strip()
        
        if source.lower() in ['nan', 'none', 'bilinmeyen', '']:
            source = f"Kaynak {used_count + 1}"
        
        source_names.append(source)
        context_chunk = f"[Bağlam {used_count + 1}]\nKaynak: {source}\nMetin: {chunk}\n\n"
        
        if len(context_text + context_chunk) > 2000:
            break
        
        context_text += context_chunk
        used_count += 1
    
    user_prompt = f"""Soru: {query}

{context_text}

Yukarıdaki bağlam bilgisini kullanarak soruyu cevapla. Yanıtın sonuna destek kaynakları ekle.
Bağlam yetersizse: "Verilen bağlamda yeterli bilgi yoktur." de."""
    
    full_prompt = f"""<s>[INST] <<SYS>>
{system_instruction}
<</SYS>>

{user_prompt} [/INST]"""
    
    return full_prompt, source_names


def generate_answer(query: str, context_chunks: pd.DataFrame) -> str:
    """Generate answer using LLM."""
    prompt, sources = build_rag_prompt(query, context_chunks)
    
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)
    
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=True if TEMPERATURE > 0 else False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = generated_text[len(prompt):].strip()
    
    return answer, sources


print("Answer generation functions defined.")

# Section 9: Evaluation Metrics Functions

In [ ]:
# ===================== HELPER FUNCTIONS FOR ROBUST TEXT HANDLING =====================

def normalize_text(text: str) -> str:
    """Safely normalize text for comparison."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    return str(text).lower().strip()


def safe_listify(value) -> List[str]:
    """
    Safely convert value to list of strings.
    Handles None, NaN, strings, lists, and JSON-like strings.
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    
    if isinstance(value, list):
        return [str(v).strip() for v in value if v is not None and not (isinstance(v, float) and np.isnan(v))]
    
    # Handle JSON string lists
    if isinstance(value, str):
        value = value.strip()
        if value.startswith('[') and value.endswith(']'):
            try:
                parsed = json.loads(value)
                if isinstance(parsed, list):
                    return [str(v).strip() for v in parsed if v is not None]
            except:
                pass
        return [value] if value else []
    
    return []


# ===================== EVALUATION METRICS FUNCTIONS =====================

def compute_rouge_l(reference: str, hypothesis: str) -> float:
    """Compute ROUGE-L score.
    
    ROUGE-L measures longest common subsequence (LCS).
    For Turkish text, we use stemming=False to preserve character structure.
    """
    try:
        # Safely convert to strings
        reference = normalize_text(reference) if reference else ""
        hypothesis = normalize_text(hypothesis) if hypothesis else ""
        
        # Handle empty cases
        if not reference or not hypothesis:
            return 0.0
        
        # Use rouge_scorer without language specification to work with Turkish
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
        scores = scorer.score(reference, hypothesis)
        
        rouge_score = scores['rougeL'].fmeasure
        return round(rouge_score, 4)
    except Exception as e:
        return 0.0


def compute_bleu(reference: str, hypothesis: str) -> float:
    """Compute BLEU score (simplified 1-gram precision)."""
    try:
        ref_tokens = normalize_text(reference).split()
        hyp_tokens = normalize_text(hypothesis).split()
        
        if len(hyp_tokens) == 0:
            return 0.0
        
        # 1-gram precision
        matches = sum(1 for token in hyp_tokens if token in ref_tokens)
        return round(matches / len(hyp_tokens), 4)
    except:
        return 0.0


def compute_token_f1(reference: str, hypothesis: str) -> float:
    """Compute token-level F1 (set-based overlap)."""
    try:
        ref_tokens = set(normalize_text(reference).split())
        hyp_tokens = set(normalize_text(hypothesis).split())
        
        if len(ref_tokens) == 0 or len(hyp_tokens) == 0:
            return 0.0
        
        intersection = len(ref_tokens & hyp_tokens)
        precision = intersection / len(hyp_tokens) if len(hyp_tokens) > 0 else 0.0
        recall = intersection / len(ref_tokens) if len(ref_tokens) > 0 else 0.0
        
        if precision + recall == 0:
            return 0.0
        
        f1 = 2 * (precision * recall) / (precision + recall)
        return round(f1, 4)
    except:
        return 0.0


def check_source_hit(retrieved_sources, gold_source: str) -> bool:
    """Check if gold source appears in retrieved sources.
    
    Handles various formats: lists, NaN, strings, JSON strings.
    Uses fuzzy matching: exact, substring, or contains.
    """
    # Normalize gold source
    gold_source_norm = normalize_text(gold_source)
    if not gold_source_norm:
        return False
    
    # Convert retrieved to list
    sources_list = safe_listify(retrieved_sources)
    
    # Check each retrieved source
    for src in sources_list:
        src_norm = normalize_text(src)
        if not src_norm:
            continue
        # Fuzzy match: exact, substring, or contains
        if (src_norm == gold_source_norm or 
            gold_source_norm in src_norm or 
            src_norm in gold_source_norm):
            return True
    
    return False


def check_article_hit(retrieved_articles, gold_article: str) -> bool:
    """Check if gold article appears in retrieved articles.
    
    Handles various formats: lists, NaN, strings, JSON strings.
    Uses fuzzy matching: exact, substring, or contains.
    """
    # Normalize gold article
    gold_article_norm = normalize_text(gold_article)
    if not gold_article_norm or gold_article_norm == 'none':
        return False
    
    # Convert retrieved to list
    articles_list = safe_listify(retrieved_articles)
    
    # Check each retrieved article
    for art in articles_list:
        art_norm = normalize_text(art)
        if not art_norm or art_norm == 'none':
            continue
        # Fuzzy match
        if (art_norm == gold_article_norm or 
            gold_article_norm in art_norm or 
            art_norm in gold_article_norm):
            return True
    
    return False


def hit_at_k(retrieved_articles, gold_article: str, k: int) -> bool:
    """Check if gold article appears in top-k retrieved."""
    articles_list = safe_listify(retrieved_articles)
    top_k_articles = articles_list[:k]
    return check_article_hit(top_k_articles, gold_article)


print("Evaluation metrics functions defined with robust text handling.")

In [ ]:
# ===================== TEST ROUGE-L WITH MANUAL EXAMPLES =====================
print("\n" + "="*80)
print("ROUGE-L TEST WITH MANUAL EXAMPLES")
print("="*80)

# Test case 1: High overlap (should be high ROUGE-L)
gold_1 = "Bireysel başvuru yapılabilmesi için olağan kanun yollarının tüketilmiş olması gerekir."
pred_1 = "Anayasa Mahkemesi'ne bireysel başvuru için olağan kanun yollarının tüketilmiş olması gerekir."
rouge_1 = compute_rouge_l(gold_1, pred_1)
print(f"\nTest 1 - High overlap (should be > 0.5):")
print(f"  Gold: {gold_1}")
print(f"  Pred: {pred_1}")
print(f"  ROUGE-L: {rouge_1} ✓" if rouge_1 > 0.3 else f"  ROUGE-L: {rouge_1} ✗ (too low!)")

# Test case 2: No overlap (should be 0)
gold_2 = "Türkiye bir cumhuriyettir."
pred_2 = "Mektep kapalı mıdır?"
rouge_2 = compute_rouge_l(gold_2, pred_2)
print(f"\nTest 2 - No overlap (should be ~0):")
print(f"  Gold: {gold_2}")
print(f"  Pred: {pred_2}")
print(f"  ROUGE-L: {rouge_2} ✓" if rouge_2 < 0.1 else f"  ROUGE-L: {rouge_2} (unexpected)")

# Test case 3: Identical (should be 1.0)
gold_3 = "Ceza muhakemesinde tutuklama şartları."
pred_3 = "Ceza muhakemesinde tutuklama şartları."
rouge_3 = compute_rouge_l(gold_3, pred_3)
print(f"\nTest 3 - Identical (should be 1.0):")
print(f"  Gold: {gold_3}")
print(f"  Pred: {pred_3}")
print(f"  ROUGE-L: {rouge_3} ✓" if rouge_3 >= 0.99 else f"  ROUGE-L: {rouge_3} (unexpected)")

print("\n" + "="*80)

# Section 10: Run Evaluation on All Gold Questions

In [ ]:
print(f"Starting evaluation on {len(gold_df)} gold questions...\n")

all_results = []

for idx, row in tqdm(gold_df.iterrows(), total=len(gold_df), desc="Evaluating"):
    gold_question = row.get('question', '')
    gold_answer = row.get('gold_answer', '')
    gold_source = row.get('gold_source', None)
    gold_article = row.get('article_reference', None)
    legal_domain = row.get('legal_domain', 'Unknown')
    difficulty = row.get('difficulty', 'Unknown')
    
    try:
        # Retrieve context
        context = get_final_context(gold_question, top_k=TOP_K_FINAL)
        
        # Generate answer
        predicted_answer, supporting_sources = generate_answer(gold_question, context)
        
        # Collect retrieved metadata
        retrieved_sources = context['source'].tolist()
        retrieved_articles = context.get('article_reference', [None] * len(context)).tolist() if 'article_reference' in context.columns else [None] * len(context)
        top_chunk_ids = context['chunk_id'].tolist()
        top_contexts = context['chunk_text'].tolist()
        
        # Compute retrieval metrics
        source_hit = check_source_hit(retrieved_sources, gold_source)
        article_hit_1 = hit_at_k(retrieved_articles, gold_article, 1)
        article_hit_3 = hit_at_k(retrieved_articles, gold_article, 3)
        article_hit_5 = hit_at_k(retrieved_articles, gold_article, 5)
        
        # Compute answer metrics
        rouge_l = compute_rouge_l(gold_answer, predicted_answer)
        bleu = compute_bleu(gold_answer, predicted_answer)
        token_f1 = compute_token_f1(gold_answer, predicted_answer)
        
        # Store result
        result = {
            'question_idx': idx,
            'question': gold_question,
            'gold_answer': gold_answer,
            'gold_source': gold_source,
            'gold_article': gold_article,
            'legal_domain': legal_domain,
            'difficulty': difficulty,
            'predicted_answer': predicted_answer,
            'retrieved_sources': json.dumps(retrieved_sources),
            'retrieved_articles': json.dumps(retrieved_articles),
            'top_chunk_ids': json.dumps(top_chunk_ids),
            'retrieval_count': len(context),
            'source_hit': source_hit,
            'article_hit@1': article_hit_1,
            'article_hit@3': article_hit_3,
            'article_hit@5': article_hit_5,
            'rouge_l': rouge_l,
            'bleu': bleu,
            'token_f1': token_f1,
            'timestamp': datetime.now().isoformat()
        }
        
        all_results.append(result)
        
    except Exception as e:
        print(f"\nError on question {idx}: {str(e)}")
        # Still save a partial result
        result = {
            'question_idx': idx,
            'question': gold_question,
            'gold_answer': gold_answer,
            'gold_source': gold_source,
            'gold_article': gold_article,
            'legal_domain': legal_domain,
            'difficulty': difficulty,
            'predicted_answer': f'ERROR: {str(e)[:100]}',
            'error': str(e)
        }
        all_results.append(result)

print(f"\n✓ Evaluation completed on {len(all_results)} questions")

# Section 11: Create Results DataFrame and Summary Metrics

In [ ]:
# Convert to DataFrame
results_df = pd.DataFrame(all_results)

print(f"Results DataFrame shape: {results_df.shape}")
print(f"\nColumns: {results_df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(results_df.head(3))

# Compute aggregate metrics
numeric_columns = ['source_hit', 'article_hit@1', 'article_hit@3', 'article_hit@5', 'rouge_l', 'bleu', 'token_f1']

summary_metrics = {}
for col in numeric_columns:
    if col in results_df.columns:
        # Handle boolean and numeric columns
        if results_df[col].dtype == bool:
            summary_metrics[f"{col}_accuracy"] = results_df[col].mean()
        else:
            summary_metrics[f"{col}_mean"] = results_df[col].mean()
            summary_metrics[f"{col}_std"] = results_df[col].std()
            summary_metrics[f"{col}_min"] = results_df[col].min()
            summary_metrics[f"{col}_max"] = results_df[col].max()

summary_metrics['total_questions'] = len(results_df)
summary_metrics['timestamp'] = datetime.now().isoformat()

print(f"\n\nAggregate Metrics:")
print(json.dumps(summary_metrics, indent=2))

# Section 12: Error Analysis

In [ ]:
# Identify problematic cases

print("\n" + "="*80)
print("ERROR ANALYSIS")
print("="*80)

# Cases where source was not retrieved
source_miss = results_df[results_df['source_hit'] == False]
print(f"\n### Source Miss Cases: {len(source_miss)} / {len(results_df)} ({100*len(source_miss)/len(results_df):.1f}%)")
if len(source_miss) > 0:
    print(source_miss[['question_idx', 'question', 'gold_source', 'retrieved_sources']].head(5).to_string())

# Cases where article was not retrieved
article_miss = results_df[results_df['article_hit@1'] == False]
print(f"\n### Article Hit@1 Miss Cases: {len(article_miss)} / {len(results_df)} ({100*len(article_miss)/len(results_df):.1f}%)")
if len(article_miss) > 0:
    print(article_miss[['question_idx', 'question', 'gold_article', 'retrieved_articles']].head(5).to_string())

# Cases with low ROUGE scores (weak answer)
low_rouge = results_df[results_df['rouge_l'] < 0.3]
print(f"\n### Low ROUGE-L Cases (< 0.3): {len(low_rouge)} / {len(results_df)} ({100*len(low_rouge)/len(results_df):.1f}%)")
if len(low_rouge) > 0:
    detail = low_rouge[['question_idx', 'question', 'predicted_answer', 'rouge_l']].copy()
    detail['predicted_answer_preview'] = detail['predicted_answer'].str[:80]
    print(detail[['question_idx', 'predicted_answer_preview', 'rouge_l']].head(5).to_string())

# Cases with high ROUGE but no source match
good_answer_bad_source = results_df[(results_df['rouge_l'] > 0.5) & (results_df['source_hit'] == False)]
print(f"\n### Good Answer But Bad Source: {len(good_answer_bad_source)} cases")
print("  (Possible hallucination without citation)")
if len(good_answer_bad_source) > 0:
    print(good_answer_bad_source[['question_idx', 'question', 'predicted_answer', 'gold_source', 'rouge_l']].head(3).to_string())

# Distribution by domain
print(f"\n### Performance by Legal Domain:")
domain_stats = results_df.groupby('legal_domain')[['rouge_l', 'token_f1', 'source_hit']].agg(['mean', 'std', 'count'])
print(domain_stats)

# Distribution by difficulty
print(f"\n### Performance by Difficulty:")
if 'difficulty' in results_df.columns:
    diff_stats = results_df.groupby('difficulty')[['rouge_l', 'token_f1', 'source_hit']].agg(['mean', 'count'])
    print(diff_stats)

print("\n" + "="*80)

# Section 13: Save Results

In [ ]:
# Save detailed results to CSV
results_csv_path = os.path.join(EVAL_OUTPUT_DIR, "kaggle_gold50_rag_results.csv")
results_df.to_csv(results_csv_path, index=False)
print(f"✓ Saved detailed results: {results_csv_path}")

# Save summary metrics to JSON
summary_json_path = os.path.join(EVAL_OUTPUT_DIR, "kaggle_gold50_metrics_summary.json")
with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(summary_metrics, f, indent=2, ensure_ascii=False)
print(f"✓ Saved summary metrics: {summary_json_path}")

# Save configuration for reference
config_dict = {
    'gold_benchmark_path': GOLD_BENCHMARK_PATH,
    'retrieval_config': {
        'top_k_final': TOP_K_FINAL,
        'dense_candidates': DENSE_CANDIDATES,
        'bm25_candidates': BM25_CANDIDATES,
        'hybrid_candidates': HYBRID_CANDIDATES,
        'alpha': ALPHA
    },
    'models': {
        'embedding': EMBEDDING_MODEL_NAME,
        'reranker': RERANKER_MODEL_NAME,
        'generation': GENERATION_MODEL_NAME
    },
    'evaluation_timestamp': datetime.now().isoformat()
}

config_json_path = os.path.join(EVAL_OUTPUT_DIR, "kaggle_gold50_evaluation_config.json")
with open(config_json_path, 'w', encoding='utf-8') as f:
    json.dump(config_dict, f, indent=2, ensure_ascii=False)
print(f"✓ Saved configuration: {config_json_path}")

print(f"\n✓ All outputs saved to: {EVAL_OUTPUT_DIR}")

# Section 14: Overall Summary Statistics

In [ ]:
print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)

print(f"""
📊 Overall Results:
   Questions evaluated: {len(results_df)}
   Errors encountered: {(results_df['predicted_answer'].str.contains('ERROR', case=False, na=False)).sum()}

🔍 Retrieval Metrics:
   Source Hit: {results_df['source_hit'].mean():.1%}
   Article Hit@1: {results_df['article_hit@1'].mean():.1%}
   Article Hit@3: {results_df['article_hit@3'].mean():.1%}
   Article Hit@5: {results_df['article_hit@5'].mean():.1%}

📝 Answer Quality Metrics:
   ROUGE-L (mean): {results_df['rouge_l'].mean():.3f} (std: {results_df['rouge_l'].std():.3f})
   BLEU (mean): {results_df['bleu'].mean():.3f} (std: {results_df['bleu'].std():.3f})
   Token F1 (mean): {results_df['token_f1'].mean():.3f} (std: {results_df['token_f1'].std():.3f})

📋 Data Distribution:
   Avg retrieved chunks per question: {results_df['retrieval_count'].mean():.1f}
   Legal domains covered: {results_df['legal_domain'].nunique()}

📁 Output Files:
   - {os.path.basename(results_csv_path)}
   - {os.path.basename(summary_json_path)}
   - {os.path.basename(config_json_path)}
"")

print("="*80)
print(f"✓ Evaluation completed at {datetime.now().isoformat()}")
print("="*80)

# Section 15: Detailed Case Studies (Optional)

In [ ]:
# Show examples of best and worst performing questions

print("\n" + "="*80)
print("BEST PERFORMING QUESTIONS (Highest ROUGE-L)")
print("="*80)

best_cases = results_df.nlargest(3, 'rouge_l')
for idx, row in best_cases.iterrows():
    print(f"\n[Question {row['question_idx']}] Domain: {row['legal_domain']}, ROUGE-L: {row['rouge_l']:.3f}")
    print(f"Q: {row['question'][:100]}...")
    print(f"Gold Answer: {row['gold_answer'][:150]}")
    print(f"Predicted: {row['predicted_answer'][:150]}")
    print(f"Source Hit: {row['source_hit']}, Article Hit@1: {row['article_hit@1']}")

print("\n" + "="*80)
print("WORST PERFORMING QUESTIONS (Lowest ROUGE-L)")
print("="*80)

worst_cases = results_df.nsmallest(3, 'rouge_l')
for idx, row in worst_cases.iterrows():
    print(f"\n[Question {row['question_idx']}] Domain: {row['legal_domain']}, ROUGE-L: {row['rouge_l']:.3f}")
    print(f"Q: {row['question'][:100]}...")
    print(f"Gold Answer: {row['gold_answer'][:150]}")
    print(f"Predicted: {row['predicted_answer'][:150]}")
    print(f"Gold Source: {row['gold_source']}")
    print(f"Source Hit: {row['source_hit']}, Article Hit@1: {row['article_hit@1']}")